In [ ]:
# Imports dan pengecekan GPU
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import pathlib

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Path dataset dan parameter
BASE_DIR = 'dataset'  # atur ke /kaggle/input/<dataset-name> saat di Kaggle
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'val')
TEST_DIR = os.path.join(BASE_DIR, 'test')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # sesuaikan jika OOM
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# Tambahan penting: Bersihkan file korup (Penyebab error Invalid Argument Format) di Kaggle Anda!
import os
from PIL import Image

def remove_corrupted_images(folder_path):
    print("Memindai gambar rusak...")
    num_deleted = 0
    # Berjalan di dalam folder training, val, test
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # Buka file untuk cek apakah beneran valid secara format bytes.
                with Image.open(file_path) as img:
                    img.verify()
            except BaseException as e:
                # Jika error atau terdeteksi bukan gambar (biasanya ._DS_Store atau icon), hapus!
                print(f"Menghapus gambar korup/file tak valid: {file_path}")
                os.remove(file_path)
                num_deleted += 1
    print(f"Pembersihan Selesai! {num_deleted} file tak valid telah dihapus.")

# Jalankan fungsi pada direktori utama dataset (Base Dir)
remove_corrupted_images(BASE_DIR)

In [ ]:
# Buat dataset pipeline
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print('Classes ({}):'.format(NUM_CLASSES), class_names)

In [ ]:
# Optimizations dan augmentasi
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [ ]:
# Bangun model transfer learning (EfficientNetB0)
IMG_SHAPE = IMG_SIZE + (3,)
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
    )
base_model.trainable = False  # freeze awal

inputs = keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Callbacks
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint('best_model.h5',
                                                 save_best_only=True,
                                                 monitor='val_accuracy',
                                                 mode='max')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                 patience=3, min_lr=1e-6)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                              restore_best_weights=True)

# Training awal (feature extraction)
EPOCHS = 15
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb, reduce_lr, early_stop]
    )

In [ ]:
# Fine-tuning beberapa layer akhir
base_model.trainable = True
fine_tune_at = 150  # sesuaikan jika perlu
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
FT_EPOCHS = 10
total_epochs = EPOCHS + FT_EPOCHS
history_f = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] if hasattr(history, 'epoch') else 0,
    callbacks=[checkpoint_cb, reduce_lr, early_stop]
    )

In [ ]:
# Plot loss & accuracy
def plot_history(h, title='Training history'):
    df = pd.DataFrame(h.history)
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    axes[0].plot(df['loss'], label='train')
    axes[0].plot(df['val_loss'], label='val')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[1].plot(df['accuracy'], label='train')
    axes[1].plot(df['val_accuracy'], label='val')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    plt.suptitle(title)
    plt.show()

plot_history(history, 'Feature extraction')
plot_history(history_f, 'Fine-tuning')

In [ ]:
# Confusion matrix dan classification report pada val set
y_true = np.array([]).astype('int32')
y_pred = np.array([]).astype('int32')
for images, labels in val_ds:
    preds = model.predict(images)
    preds_idx = np.argmax(preds, axis=1)
    y_true = np.concatenate([y_true, labels.numpy()])
    y_pred = np.concatenate([y_pred, preds_idx])

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (validation)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))
print('Macro F1:', f1_score(y_true, y_pred, average='macro'))

In [ ]:
# Simpan model dan buat prediksi pada test set (jika ada)
model.save('final_model')
# Siapkan test dataset: jika test memiliki subfolders label -> labels='inferred',
# jika test tidak berlabel gunakan labels=None dan shuffle=False untuk mapping nama file ke prediksi
test_subdirs = [p for p in pathlib.Path(TEST_DIR).iterdir() if p.is_dir()]
if len(test_subdirs) > 0:
    test_ds = tf.keras.utils.image_dataset_from_directory(TEST_DIR, labels='inferred', image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
    loss, acc = model.evaluate(test_ds)
    print('Test loss, accuracy:', loss, acc)
else:
    images = list(pathlib.Path(TEST_DIR).rglob('*.jpg')) + list(pathlib.Path(TEST_DIR).rglob('*.png'))
    images = sorted(images)
    file_paths = [str(p) for p in images]
    def load_and_preprocess(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        img = tf.keras.applications.efficientnet.preprocess_input(img)
        return img
    test_images = tf.data.Dataset.from_tensor_slices(file_paths)
    test_images = test_images.map(lambda x: load_and_preprocess(x), num_parallel_calls=AUTOTUNE)
    test_images = test_images.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    preds = model.predict(test_images)
    preds_idx = np.argmax(preds, axis=1)
    label_map = {i: n for i, n in enumerate(class_names)}
    results = pd.DataFrame({'file': file_paths, 'pred': [label_map[i] for i in preds_idx]})
    results.to_csv('test_predictions.csv', index=False)
    print('Saved test_predictions.csv with', len(results), 'rows')

In [ ]:
# Ekspor Metadata untuk keperluan Backend
import json
import os

print(\"Menyimpan Metadata...\")
metadata = {
    \"model_name\": \"efficientnet_b0\",
    \"num_classes\": NUM_CLASSES,
    \"img_size\": 224,
    \"class_names\": class_names
}

with open(\"model_metadata.json\", \"w\") as f:
    json.dump(metadata, f, indent=2)
    
print(\"Model 'best_model.h5', 'final_model', dan 'model_metadata.json' siap dipakai di backend!\")

## Catatan akhir
- Sesuaikan `BATCH_SIZE` jika OOM di GPU.
- Jika dataset sangat besar, pertimbangkan `tf.data` pipeline tambahan dan mixed precision.
- Untuk submission Kaggle, cek format yang diminta dan sesuaikan `test_predictions.csv`.